# P22 — Prédire les inspections sanitaires à risque à Chicago (Food Inspections)

**Contexte métier.** Le département de la santé de Chicago réalise ~15 000 inspections d'établissements alimentaires par an. Chaque inspection aboutit à Pass / Fail / Pass with Conditions. Anticiper les établissements à risque de « Fail » permet de prioriser les visites et les accompagnements des exploitants avant l'inspection.

**Objectif de l'analyse exploratoire.** Identifier la cible (résultat Fail) et les régresseurs (type d'établissement, type d'inspection, historique de l'établissement).

**Données.** Chicago Data Portal « Food Inspections » (Socrata), 200 000 inspections récentes (2015-2026).

## 1. Téléchargement

Export Socrata de la ville de Chicago. La colonne `violations` (texte) pèse l'essentiel des 226 Mo : on la charge séparément si nécessaire.

In [ ]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
URL = "https://data.cityofchicago.org/resource/4ijn-s7e5.csv?$limit=200000"
if not os.path.exists("data/chicago_food.csv"):
    urllib.request.urlretrieve(URL, "data/chicago_food.csv")
print("OK", os.path.getsize("data/chicago_food.csv") // (1024 * 1024), "Mo")

## 2. Chargement

Chargement sélectif (sans `violations`) : inspecter dimensions et manquants, puis construire la cible.

In [ ]:
import pandas as pd

usecols = ["inspection_id", "dba_name", "facility_type", "risk", "city", "inspection_date",
           "inspection_type", "results", "latitude", "longitude", "license_"]
df = pd.read_csv("data/chicago_food.csv", usecols=usecols, low_memory=False)
print(df.shape, "lignes x colonnes chargées")
manq = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("% manquant :")
print(manq[manq > 0].to_string())

df["insp_dt"] = pd.to_datetime(df["inspection_date"], errors="coerce")
df["fail"] = df["results"] == "Fail"
print()
print("Période :", df["insp_dt"].min().date(), "->", df["insp_dt"].max().date())
print("Résultats :")
print(df["results"].value_counts(normalize=True).mul(100).round(1).to_string())

## 3. Analyse

Taux de Fail par type d'établissement et par type d'inspection ; effet de l'historique (échec précédent de la même licence).

In [ ]:
print("Types d'établissement (top 8) :")
print(df["facility_type"].value_counts().head(8).to_string())
print()
g = df.groupby("facility_type")["fail"].agg(["mean", "count"])
g = g[g["count"] > 3000].sort_values("mean", ascending=False)
print("Taux de Fail par type (au moins 3 000 inspections) :")
print(g["mean"].mul(100).round(1).to_string())
print()
print("Taux de Fail par type d'inspection :")
print(df.groupby("inspection_type")["fail"].agg(["mean", "count"])
      .query("count > 3000")["mean"].mul(100).round(1).sort_values(ascending=False).to_string())
print()
prev = df.sort_values("insp_dt").groupby("license_")["fail"].shift(1)
df["fail_prec"] = prev
sub = df.dropna(subset=["fail_prec"])
print("Effet historique : accord entre le Fail et le résultat précédent de la même licence :",
      round(100 * (sub["fail"] == sub["fail_prec"]).mean(), 1), "%")

## 4. Résultats observés (exécution du 16/10/2026)

- 200 000 inspections, 17 colonnes brutes (13 chargées) ; période 24/11/2015 -> 15/10/2026
- résultats : Pass 47,3 %, Fail 19,1 %, Pass w/ Conditions 18,2 %, Out of Business 7,3 %, No Entry 5,8 %
- cible Fail : 19,1 % de positifs (classification modérément déséquilibrée)
- établissements : Restaurant 139 760 (70 %), Grocery Store 21 557, School 12 293 ; 24 597 noms, 33 185 licences
- taux de Fail par type : School 24,5 %, Grocery Store 21,0 %, Restaurant 18,5 %, Children's Services 16,4 %
- par type d'inspection : Complaint 33,5 %, License 23,5 %, Canvass 19,9 %, Canvass Re-Inspection 10,0 %
- manquants quasi nuls sur les colonnes chargées (facility_type 0,5 %, lat/lon 0,3 %)
- historique : 166 808 inspections avec inspection précédente ; accord Fail/Fail précédent : 63,9 %

## 5. Variable cible et régresseurs

**Cible (classification binaire) : inspection « Fail »** — 19,1 % de positifs. Baseline honnête : prédire l'échec précédent de la même licence (accord 63,9 %).

**Régresseurs retenus :**
- `facility_type` (School 24,5 % vs Restaurant 18,5 %), `risk` (1/2/3)
- historique de la licence (`license_`) : échec précédent, nombre d'inspections passées, délai depuis la dernière
- `inspection_type` (Complaint 33,5 % vs Canvass 19,9 %) — mais le type révèle déjà une suspicion : à discuter en fuite
- `mois`, ancienneté de l'établissement
- enrichissement possible : la colonne `violations` (texte des codes de violation) pour un modèle NLP

**Pièges :**
- `results` compte 7 modalités (Out of Business, No Entry, Not Ready...) : exclure les inspections sans résultat exploitable avant de binariser
- une inspection « Complaint » est déclenchée par une plainte : le type d'inspection encode une information post-érieure à l'état de l'établissement — à choisir consciemment comme feature ou à retirer selon l'usage
- un même établissement apparaît plusieurs fois : le split train/test doit se faire par licence, pas par ligne
- les reinspections suivent un échec (10 % de Fail seulement) : modéliser inspection initiale et réinspection séparément
- la ville publie les résultats avec du retard : les 200 000 dernières lignes ne sont pas les 200 000 dernières années